# PAA KX2

The [PAA KX2](https://www.peakanalysisandautomation.com/products/plate-handlers/kx2/) (also sold as KiNEDx KX2) is a 5-axis SCARA plate-handling robot from Peak Analysis and Automation. It supports:

- [Arms](../../capabilities/arms) (pick/place, Cartesian and joint movement, freedrive teaching)

The device communicates over CAN bus using the CANopen protocol (CiA-301 + CiA-402 drive profile). A PCAN/SocketCAN-compatible CAN interface is required (e.g. PEAK PCAN-USB).

| Model | PLR Name | Gripper | Notes |
|---|---|---|---|
| KX2 | `KX2` | servo | 4 motion axes (shoulder, Z, elbow, wrist) + servo gripper |

## Setup

`setup()` opens the CAN bus, discovers nodes, reads drive parameters, enables the motion axes, and homes + closes the servo gripper.

In [2]:
from pylabrobot.paa.kx2 import KX2

kx2 = KX2()
await kx2.setup()

uptime library not available, timestamps are relative to boot time and not to Epoch UTC
2026-05-03 19:56:58,933 - pylabrobot.paa.kx2.driver - INFO - canopen: connected, nodes=[1, 2, 3, 4, 6]
2026-05-03 19:57:01,871 - pylabrobot.paa.kx2.driver - WARNING - node 1: re-enabling (was disabled)
2026-05-03 19:57:02,082 - pylabrobot.paa.kx2.driver - WARNING - motor_enable(state=True) attempt 1/20 failed for node 1 (MO=0); retrying
2026-05-03 19:57:02,704 - pylabrobot.paa.kx2.driver - WARNING - node 3: re-enabling (was disabled)
2026-05-03 19:57:02,835 - pylabrobot.paa.kx2.driver - WARNING - node 4: re-enabling (was disabled)


## Arm capabilities

The KX2 exposes an {class}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm` on `kx2.arm`, backed by {class}`~pylabrobot.paa.kx2.KX2ArmBackend`. Gripper yaw is controlled via a single `direction` float (degrees; 0° = front). For the full arm API, see [Arms](../../capabilities/arms).

The raw driver (`kx2.driver`, a `KX2Driver`) stays available for low-level access — motor commands, drive diagnostics, binary interpreter, etc.

### Gripper control

The servo gripper is force-aware. `close_gripper` moves to the target width and then verifies the plate is gripped; pass `check_plate_gripped=False` via `GripParams` to skip the verification move. Units are KX2 gripper encoder units (mm-equivalent once calibrated).

In [4]:
await kx2.arm.open_gripper(gripper_width=25)

In [5]:
from pylabrobot.paa.kx2 import KX2ArmBackend

await kx2.arm.close_gripper(
    gripper_width=0,
    backend_params=KX2ArmBackend.GripParams(check_plate_gripped=False),
)

In [6]:
await kx2.arm.is_gripper_closed()

True

Cap the gripping force on a per-close basis (as a percentage of the motor's peak current — 10..100, clamped) when closing on fragile labware:

In [7]:
await kx2.arm.close_gripper(
    gripper_width=0,
    backend_params=KX2ArmBackend.GripParams(max_force_percent=30),
)

### Motion limits

`motion_limits()` reads the per-axis firmware maxima cached at setup. `JointMoveParams` / `CartesianMoveParams` take `max_gripper_speed` (mm/s) and `max_gripper_acceleration` (mm/s^2), which cap the worst-case Cartesian speed/acceleration at the gripper along the trajectory. `None` (the default) means no Cartesian cap — joints run at firmware max. Joint speeds get scaled uniformly so the gripper stays under the cap.

In [8]:
kx2.arm.backend.motion_limits()

axis           max speed     max acceleration
-------------  ------------  ----------------
SHOULDER       145.00 deg/s  300.00 deg/s^2  
Z              750.00 mm/s   1000.00 mm/s^2  
ELBOW          80.00 deg/s   180.00 deg/s^2  
WRIST          500.00 deg/s  1000.04 deg/s^2 
SERVO_GRIPPER  47.00 mm/s    200.00 mm/s^2   

### Cartesian movement

Move the arm to a Cartesian location. `direction` is the gripper yaw in degrees (Z-axis rotation only — the KX2 cannot roll or pitch). Use {class}`~pylabrobot.paa.kx2.KX2ArmBackend.CartesianMoveParams` to override velocity and acceleration.

In [9]:
await kx2.arm.request_gripper_location()

GripperLocation(location=Coordinate(x=312.2642, y=564.2383, z=333.7532), rotation=Rotation(x=0, y=0, z=265.67006324798695))

In [15]:
from pylabrobot.resources import Coordinate

await kx2.arm.move_to_location(
    location=Coordinate(x=330.124, y=210.552, z=320.0441),
    direction=198,
    backend_params=KX2ArmBackend.CartesianMoveParams(
        max_gripper_speed=200.0, max_gripper_acceleration=1000.0,
    ),
)

In [ ]:
from pylabrobot.resources import Coordinate

await kx2.arm.move_to_location(
    location=Coordinate(x=130.124, y=210.552, z=420.0441),
    direction=100,
    backend_params=KX2ArmBackend.CartesianMoveParams(
      max_gripper_speed=200.0,
      max_gripper_acceleration=1000.0,
      path="joint"
    ),
)

### Joint movement

Move in joint space using the {class}`~pylabrobot.paa.kx2.KX2ArmBackend.Axis` enum. Rotary axes in degrees; Z (and gripper) in mm-equivalent encoder units.

```{warning}
Moving to arbitrary joint positions can cause the arm to collide with its base, gripper, or nearby equipment. Verify coordinates in a safe workspace first, and start with low velocity.
```

In [1]:
from pylabrobot.paa.kx2 import KX2ArmBackend

position = {
    KX2ArmBackend.Axis.SHOULDER: 0.0,
    KX2ArmBackend.Axis.Z: 150.0,
    KX2ArmBackend.Axis.ELBOW: 90.0,
    KX2ArmBackend.Axis.WRIST: 0.0,
}
await kx2.arm.backend.move_to_joint_position(
    position,
    backend_params=KX2ArmBackend.JointMoveParams(
        max_gripper_speed=200.0, max_gripper_acceleration=1000.0,
    ),
)

AttributeError: type object 'KX2ArmBackend' has no attribute 'Axis'

### Joint-space pick and place

`pick_up_at_joint_position` / `drop_at_joint_position` do the same thing as their Cartesian counterparts (move, then close/open the gripper), but the target pose is specified in joint space. Useful when you've taught a position via freedrive and want to return to it without going through inverse kinematics.

In [ ]:
pick_joints = {
    KX2ArmBackend.Axis.SHOULDER: 0.0,
    KX2ArmBackend.Axis.Z: 150.0,
    KX2ArmBackend.Axis.ELBOW: 90.0,
    KX2ArmBackend.Axis.WRIST: 0.0,
}
place_joints = {
    KX2ArmBackend.Axis.SHOULDER: 30.0,
    KX2ArmBackend.Axis.Z: 150.0,
    KX2ArmBackend.Axis.ELBOW: 90.0,
    KX2ArmBackend.Axis.WRIST: 0.0,
}

await kx2.arm.backend.pick_up_at_joint_position(pick_joints, resource_width=0)
await kx2.arm.backend.drop_at_joint_position(place_joints, resource_width=30)

### Querying position

Read the current joint angles or Cartesian end-effector pose:

In [ ]:
await kx2.arm.backend.request_joint_position()

In [ ]:
await kx2.arm.request_gripper_location()

### Pick and place

`pick_up_at_location` moves to the target pose and closes the gripper to `resource_width`. `drop_at_location` moves and opens the gripper. Approach and retract motions are the caller's responsibility — bracket these calls with your own pre- and post-moves.

In [ ]:
pick  = Coordinate(x=450.0, y=0.0,    z=80.0)
place = Coordinate(x=450.0, y=-200.0, z=80.0)
above = Coordinate(x=0,     y=0,      z=60.0)
yaw   = 0.0

# pick_up_at_location stores resource_width on the frontend; drop_at_location
# reuses it, so you don't pass it again.
await kx2.arm.move_to_location(pick + above, direction=yaw)
await kx2.arm.pick_up_at_location(location=pick, direction=yaw, resource_width=0)
await kx2.arm.move_to_location(pick + above, direction=yaw)

await kx2.arm.move_to_location(place + above, direction=yaw)
await kx2.arm.drop_at_location(location=place, direction=yaw)
await kx2.arm.move_to_location(place + above, direction=yaw)

### Freedrive (teaching mode)

Freedrive disables torque on the motion axes so you can push the arm by hand; the servo gripper stays powered. The KX2 frees all motion axes at once — the `free_axes` argument is accepted for API parity and ignored.

In [ ]:
await kx2.arm.backend.start_freedrive_mode(free_axes=[0])

In [ ]:
# Manually guide the arm to the desired pose, then read it:
taught = await kx2.arm.request_gripper_location()
print(taught)

In [ ]:
await kx2.arm.backend.stop_freedrive_mode()

### Halt and fault diagnostics

`halt` issues an emergency stop on every motion axis. Driver-level methods give you estop-line state and per-axis fault codes.

In [ ]:
await kx2.arm.backend.halt()

In [ ]:
await kx2.driver.get_estop_state()

In [ ]:
from pylabrobot.paa.kx2 import KX2ArmBackend

await kx2.driver.motor_get_fault(KX2ArmBackend.Axis.SHOULDER)

## Teardown

In [ ]:
await kx2.stop()